# Solar Filament Segmentation - fixed submission pipeline

Version 19 fixed pipeline. Empty masks are skipped, official submission columns are used, compressed COCO RLE is generated at 2048 x 2048, and the CSV is validated after a round trip.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations pycocotools opencv-python tqdm

from pathlib import Path
import glob
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SIZE = (512, 512)
OUTPUT_SIZE = (2048, 2048)
THRESHOLD = 0.50
BATCH_SIZE = 8
NUM_WORKERS = 2
OUTPUT_PATH = '/kaggle/working/submission_v19_fixed.csv'

search_roots = glob.glob('/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026', recursive=True)
BASE_DATA_DIR = search_roots[0] if search_roots else '/kaggle/input'
TEST_IMG_DIR = os.path.join(BASE_DATA_DIR, 'test', 'test_images')
if not os.path.isdir(TEST_IMG_DIR):
    raise FileNotFoundError(f'Test image directory not found: {TEST_IMG_DIR}')

CHECKPOINT_PATHS = sorted(glob.glob('/kaggle/input/**/best_solar_unet.pth', recursive=True))
if not CHECKPOINT_PATHS:
    raise FileNotFoundError('No best_solar_unet.pth found under /kaggle/input.')
print({'device': str(DEVICE), 'test_dir': TEST_IMG_DIR, 'checkpoints': CHECKPOINT_PATHS})

In [ ]:
class SolarTestDataset(Dataset):
    def __init__(self, image_dir, image_size=INPUT_SIZE):
        extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
        self.paths = sorted(path for ext in extensions for path in glob.glob(os.path.join(image_dir, ext)))
        if not self.paths:
            raise FileNotFoundError(f'No test images found in {image_dir}')
        self.transform = A.Compose([
            A.Resize(*image_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = cv2.imread(path, cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f'OpenCV could not read image: {path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        tensor = self.transform(image=image)['image']
        return tensor, os.path.basename(path)

def build_model():
    return smp.UnetPlusPlus(encoder_name='efficientnet-b3', encoder_weights=None, in_channels=3, classes=1)

def load_model(checkpoint_path):
    model = build_model()
    payload = torch.load(checkpoint_path, map_location='cpu')
    if isinstance(payload, dict) and 'state_dict' in payload:
        payload = payload['state_dict']
    if isinstance(payload, dict) and 'model_state_dict' in payload:
        payload = payload['model_state_dict']
    if not isinstance(payload, dict):
        raise TypeError(f'Unsupported checkpoint format: {checkpoint_path}')
    state = {key.removeprefix('module.').removeprefix('model.'): value for key, value in payload.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        raise RuntimeError(f'Checkpoint mismatch: missing={missing[:3]}, unexpected={unexpected[:3]}')
    return model.to(DEVICE).eval()

models = [load_model(path) for path in CHECKPOINT_PATHS]
test_dataset = SolarTestDataset(TEST_IMG_DIR)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))
print(f'Loaded {len(models)} model(s) and {len(test_dataset)} test images.')

In [ ]:
@torch.inference_mode()
def predict_masks(loader, model_list):
    names, masks = [], []
    for images, batch_names in tqdm(loader, desc='Inference'):
        images = images.to(DEVICE, non_blocking=True)
        probabilities = [torch.sigmoid(model(images)) for model in model_list]
        probability = torch.stack(probabilities, dim=0).mean(dim=0)[:, 0].cpu().numpy()
        for name, prob in zip(batch_names, probability):
            prob_2048 = cv2.resize(prob, OUTPUT_SIZE, interpolation=cv2.INTER_LINEAR)
            masks.append((prob_2048 >= THRESHOLD).astype(np.uint8))
            names.append(name)
    return names, masks

def encode_coco_rle(binary_mask):
    binary_mask = np.asarray(binary_mask, dtype=np.uint8)
    if binary_mask.shape != OUTPUT_SIZE:
        raise ValueError(f'Expected mask shape {OUTPUT_SIZE}, got {binary_mask.shape}')
    if not np.any(binary_mask):
        return None
    encoded = mask_utils.encode(np.asfortranarray(binary_mask))
    counts = encoded['counts']
    return counts.decode('ascii') if isinstance(counts, bytes) else str(counts)

def build_submission(image_names, masks, output_path):
    rows = []
    for image_name, binary_mask in zip(image_names, masks):
        rle = encode_coco_rle(binary_mask)
        if rle is None or not rle.strip():
            continue
        rows.append({'filament_id': f'{Path(image_name).stem}_1', 'segmentation_rle': rle})
    submission = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
    if submission.empty:
        raise ValueError('No non-empty predictions were generated.')
    if submission.isna().any().any():
        raise ValueError('Submission contains null values.')
    if submission['filament_id'].duplicated().any():
        raise ValueError('Submission contains duplicate filament_id values.')
    if submission['segmentation_rle'].astype(str).str.strip().eq('').any():
        raise ValueError('Submission contains an empty segmentation_rle value.')
    submission.to_csv(output_path, index=False, lineterminator='\n')
    return submission

image_names, predicted_masks = predict_masks(test_loader, models)
submission = build_submission(image_names, predicted_masks, OUTPUT_PATH)
print(f'Wrote {len(submission)} non-empty predictions to {OUTPUT_PATH}')

In [ ]:
def validate_submission(path):
    frame = pd.read_csv(path, keep_default_na=False)
    expected = ['filament_id', 'segmentation_rle']
    if frame.columns.tolist() != expected:
        raise ValueError(f'Expected columns {expected}, got {frame.columns.tolist()}')
    if frame.empty or frame.isna().any().any():
        raise ValueError('Submission is empty or contains null values after CSV round-trip.')
    if frame['filament_id'].astype(str).str.strip().eq('').any():
        raise ValueError('Submission contains an empty filament_id.')
    if frame['segmentation_rle'].astype(str).str.strip().eq('').any():
        raise ValueError('Submission contains an empty segmentation_rle.')
    if frame['filament_id'].duplicated().any():
        raise ValueError('Submission contains duplicate filament_id values.')
    for value in frame['segmentation_rle']:
        decoded = mask_utils.decode({'size': list(OUTPUT_SIZE), 'counts': value.encode('ascii')})
        if decoded.shape != OUTPUT_SIZE:
            raise ValueError(f'Invalid decoded mask shape: {decoded.shape}')
    return frame

validated = validate_submission(OUTPUT_PATH)
print('VALIDATED:', OUTPUT_PATH)
print(validated.head())